In [47]:
import random

# Problems with prev :
# We need to improve upon previous, since it will get highly
# complicated to develop longer or more intricate chains
# with prev thing and we will need to design new class functions
# for each new type of chain.

# Solution :
# Hence we need to standardize components, such that
# We can execute them one after other. This creates Lego Like
# customizability to build pipelinable chains

# To standardize components, We need to convert them into
# runnables which will contain methods like invoke.
# We can define an abstract class Runnable, which is
# inherited by all components, such that they will need
# to implement common methods like invoke()
# Hence maintain a common structure.



In [48]:
# define abstract runnable class
from abc import ABC, abstractclassmethod

class Runnable(ABC):
    @abstractclassmethod
    def invoke(input_dict):
        pass
    

In [49]:
# Now components need to inherit Runnable.
# And must define the invoke() fun, otherwise error is thrown.

class DummyLLM(Runnable):
    def __init__(self) -> None:
        print("LLM Created")

    def invoke(self, prompt):
        # response_list = ["Delhi is capital of India",
        #                 "Sachin is my name",
        #                 "Sky has darkish blue color"]
        response = f"---Echoing dummy LLM response--- \n {prompt}"
        return {"content":response}
    
    # Note that invoke does the same thing as predict
    def predict(self, prompt):
        response_list = ["Delhi is capital of India",
                         "Sachin is my name",
                         "Sky has darkish blue color"]
        
        return {"content":random.choice(response_list)}
    

In [50]:
# Prompt template, note that they have format() method
# We will make Runnable inherited, implement invoke()

class DummyPromptTemplate(Runnable):
    def __init__(self, template, input_variables) -> None:
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict):
        return self.template.format(**input_dict)

    def format(self, input_dict):
        # self.template is actually a sort of f-string
        # passed, hence we need to format it by passing input_dict
        # contents which are passed in template.format()
        return self.template.format(**input_dict)

In [51]:
# We now need to define a class, that will connect
# multiple components together, However long they are
# Note that this class itself will be Runnable to allow
# lego like connections

class RunnableConnector(Runnable):

    # When initializing it, we pass the list of components
    # We need to connect together
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_dict):
        # We can run loop on runnables list, and pass the
        # input data into the chain

        for runnable in self.runnable_list:
            input_dict = runnable.invoke(input_dict)
        return input_dict

In [52]:
# Lets test it

llm = DummyLLM()

template = DummyPromptTemplate(
    template= "Write a {length} poem about {topic}",
    input_variables= ["length", "topic"]
)

chain = RunnableConnector([template,llm])

response = chain.invoke({"length":"short","topic":"sad"})

LLM Created


In [58]:
print(response["content"])

---Echoing dummy LLM response--- 
 Write a short poem about sad


In [59]:
# It works, We can hence connect any length of chains
# But before that, We need to define a simple parser, which
# gets string content out of prev llm's output

class DummyStrParser(Runnable):
    def invoke(self, llm_output):
        return llm_output["content"]
    
parser = DummyStrParser()

In [62]:
chain2 = RunnableConnector([template, llm, parser, llm, parser, llm, parser])

response = chain2.invoke({"length":"short","topic":"sad"})

print(response)

---Echoing dummy LLM response--- 
 ---Echoing dummy LLM response--- 
 ---Echoing dummy LLM response--- 
 Write a short poem about sad
